In [9]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt')# 
dataset.fetch_dataset(name = "CMtMedQA_ten", dataset_path = "/hongyi/stream/dataset/paper_data", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)

2025-06-30 12:00:36.670 | INFO     | stream_topic.utils.dataset:fetch_dataset:119 - Fetching dataset: CMtMedQA_ten
2025-06-30 12:00:36.671 | INFO     | stream_topic.utils.dataset:fetch_dataset:129 - Fetching dataset from local path
Preprocessing documents: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 48413/48413 [02:32<00:00, 318.17it/s]


In [10]:
all_words = [word for tokens in dataset.dataframe['tokens'] for word in tokens]
total_tokens = len(all_words)
print("整个数据集的 token 总数：", total_tokens)

整个数据集的 token 总数： 8042781


In [22]:
from stream_topic.metrics import ISIM, INT, ISH, Expressivity, NPMI, Embedding_Coherence, Embedding_Topic_Diversity
# from sentence_transformers import SentenceTransformer
from stream_topic.metrics.metrics_config import MetricsConfig
import numpy as np
MetricsConfig.set_PARAPHRASE_embedder("/hongyi/stream/sentence-transformers/Conan-embedding-v1/")#paraphrase-multilingual-mpnet-base-v2
MetricsConfig.set_SENTENCE_embedder("/hongyi/stream/sentence-transformers/Conan-embedding-v1/")#all-mpnet-base-v2

In [36]:
model = KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1",stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt')
metric = INT()
best_params = model.optimize_and_fit(dataset=dataset,
                                     criterion="custom",
        custom_metric=metric,
        min_topics=14,
        max_topics=14,
        n_trials=20,)

Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/


[I 2025-06-29 15:06:46,360] A new study created in memory with name: no-name-da6e4c55-3de9-41b8-910b-00d78ee3d14e
2025-06-29 15:06:46.363 | WARNING  | stream_topic.commons.check_steps:check_dataset_steps:45 - The following preprocessing steps are recommended for the KmeansTM model:
lowercase,
lemmatize,
stem,
expand_contractions,
remove_accents
Include them by running: dataset.preprocess(model_type='KmeansTM')
2025-06-29 15:06:46.364 | INFO     | stream_topic.models.KmeansTM:fit:219 - --- Training KmeansTM topic model ---
2025-06-29 15:06:47.057 | INFO     | stream_topic.models.abstract_helper_models.base:prepare_embeddings:225 - --- Creating /hongyi/stream/sentence-transformers/Conan-embedding-v1 document embeddings ---
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19399/19399 [08:32<00:00, 37.85it/s]
2025-06-29 15:15:19.975 | INFO     | stream_topic.models.abstract_helper_models.base:dim_red

In [37]:
best_params

{'best_params': {'n_neighbors': 44,
  'n_components': 10,
  'metric': 'euclidean',
  'init': 'k-means++',
  'n_init': 27,
  'max_iter': 642},
 'optimal_n_topics': 14,
 'best_score': -0.64}

In [38]:
import pandas as pd
total_topics, NPMI_topics = [], []
ISIM1, INT1, ISH1, WESS1, EXPRS1, NPMI1, COH1 = [], [], [], [], [], [], []
for i in range(1):
    model = KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1",stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt', **best_params['best_params'])
    model.fit(dataset,n_topics=14, language = "chinese")#
    
    topics = model.get_topics()
    total_topics.append(topics)
    
    score_list=[]
    metric = ISIM()
    for i in range(100):    
        scores = metric.score(topics) #值越小越好
        score_list.append(scores)
    ISIM1.append(np.mean(score_list))
    score_list=[]
    metric = INT()
    for i in range(100):    
        scores = metric.score(topics) #值越大越好
        score_list.append(scores)
    INT1.append(np.mean(score_list))
    score_list=[]
    metric = ISH()
    for i in range(100):    
        scores = metric.score(topics) #值越小越好
        score_list.append(scores)
    ISH1.append(np.mean(score_list))
    beta = np.random.rand(14, 384)
    diversity_metric = Embedding_Topic_Diversity()
    scores = diversity_metric.score(topics, beta)  #值越小越好
    WESS1.append(scores)
    expressivity_metric = Expressivity(
    n_words=10,
    custom_stopwords='/hongyi/stream/stopwords/common_stopwords.txt'
    )
    scores = expressivity_metric.score(topics, beta) #值越小越好
    EXPRS1.append(scores)
    metric = NPMI(dataset,language = "chinese", stopwords='/hongyi/stream/stopwords/common_stopwords.txt') #值越大越好    
    scores = metric.score(topics)  #值越大越好
    NPMI1.append(scores)
    metric = NPMI(dataset,language = "chinese", stopwords='/hongyi/stream/stopwords/common_stopwords.txt') #值越大越好    
    scores2 = metric.score_per_topic(topics)  #值越大越好
    NPMI_topics.append(scores2)
    metric = Embedding_Coherence()
    overall_score = metric.score(topics)
    COH1.append(overall_score)

metrics = {'ISIM':ISIM1, 'INT':INT1, 'ISH':ISH1, 'WESS':WESS1, 'EXPRS':EXPRS1, 'NPMI':NPMI1, 'COH':COH1}
df = pd.DataFrame(metrics).transpose()
df.to_csv('/hongyi/STREAM/result/benchmark/KmeansTM_metrics.csv')
df2 = pd.DataFrame(total_topics) 
df2.to_csv('/hongyi/STREAM/result/benchmark/KmeansTM_topics.csv')
df3 = pd.DataFrame(NPMI_topics) 
df3.to_csv('/hongyi/STREAM/result/benchmark/KmeansTM_NPMI.csv')

2025-06-29 19:28:47.669 | INFO     | stream_topic.models.KmeansTM:fit:219 - --- Training KmeansTM topic model ---
2025-06-29 19:28:49.131 | INFO     | stream_topic.models.abstract_helper_models.base:prepare_embeddings:225 - --- Creating /hongyi/stream/sentence-transformers/Conan-embedding-v1 document embeddings ---
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19399/19399 [06:57<00:00, 46.42it/s]
2025-06-29 19:35:47.695 | INFO     | stream_topic.models.abstract_helper_models.base:dim_reduction:196 - --- Reducing dimensions ---
/hongyi/anaconda3/envs/stream2/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2025-06-29 19:35:52.898 | INFO     | stream_topic.models.KmeansTM:_clustering:158 - --- Creating document cluster ---
2025-06-29 19:35:53.152 | INFO     | stre

Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/
Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/
Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/
Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/
Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/
Loading model from local path: /hongyi/stream/sentence-transformers/Conan-embedding-v1/
